In [ ]:
from pathlib import Path
import pandas as pd

daily_path = Path("../data/daily/RELIANCE.parquet")

df_daily = pd.read_parquet(daily_path)

print(df_daily.head())
print(df_daily.info())
print(df_daily.shape)


In [ ]:
minute_path = Path("../data/minute/RELIANCE.parquet")

df_minute = pd.read_parquet(minute_path)

print(df_minute.head())
# print(df_minute.info())
print(df_minute.shape)

In [ ]:
print("Daily columns:", df_daily.columns.tolist())
print("Minute columns:", df_minute.columns.tolist())

print("Daily date range:", df_daily["date"].min(), "to", df_daily["date"].max())
print("Minute date range:", df_minute["timestamp"].min(), "to", df_minute["timestamp"].max())

In [ ]:
print("Daily missing values:")
print(df_daily.isna().sum())

print("Daily duplicate dates:", df_daily["date"].duplicated().sum())

print("Minute missing values:")
print(df_minute.isna().sum())

print("Minute duplicate timestamps:", df_minute["timestamp"].duplicated().sum())

print("Zero-volume minute bars:", (df_minute["volume"] == 0).sum())

In [ ]:
from pathlib import Path
import pandas as pd

daily_dir = Path("../data/daily")

daily_summaries = []

for file in sorted(daily_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    daily_summaries.append({
        "symbol": file.stem,
        "rows": len(df),
        "start_date": df["date"].min(),
        "end_date": df["date"].max(),
        "duplicate_dates": df["date"].duplicated().sum(),
        "missing_values": int(df.isna().sum().sum()),
        "non_positive_open": int((df["open"] <= 0).sum()),
        "non_positive_high": int((df["high"] <= 0).sum()),
        "non_positive_low": int((df["low"] <= 0).sum()),
        "non_positive_close": int((df["close"] <= 0).sum()),
        "negative_volume": int((df["volume"] < 0).sum()),
        "zero_volume_days": int((df["volume"] == 0).sum()),
    })

daily_summary = pd.DataFrame(daily_summaries)

daily_summary.head()

In [ ]:
print("Number of symbols:", daily_summary["symbol"].nunique())
print("Row count range:", daily_summary["rows"].min(), "to", daily_summary["rows"].max())
print("Overall start date:", daily_summary["start_date"].min())
print("Overall end date:", daily_summary["end_date"].max())

print("\nSymbols with duplicate dates:")
display(daily_summary[daily_summary["duplicate_dates"] > 0])

print("\nSymbols with missing values:")
display(daily_summary[daily_summary["missing_values"] > 0])

print("\nSymbols with invalid prices:")
display(
    daily_summary[
        (daily_summary["non_positive_open"] > 0)
        | (daily_summary["non_positive_high"] > 0)
        | (daily_summary["non_positive_low"] > 0)
        | (daily_summary["non_positive_close"] > 0)
    ]
)

print("\nSymbols with negative volume:")
display(daily_summary[daily_summary["negative_volume"] > 0])

In [ ]:
minute_dir = Path("../data/minute")

minute_summaries = []

for file in sorted(minute_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["date"] = df["timestamp"].dt.date

    bars_per_day = df.groupby("date").size()

    minute_summaries.append({
        "symbol": file.stem,
        "rows": len(df),
        "start_timestamp": df["timestamp"].min(),
        "end_timestamp": df["timestamp"].max(),
        "trading_days": df["date"].nunique(),
        "duplicate_timestamps": df["timestamp"].duplicated().sum(),
        "missing_values": int(df.isna().sum().sum()),
        "zero_volume_bars": int((df["volume"] == 0).sum()),
        "non_positive_prices": int(
            (
                (df["open"] <= 0)
                | (df["high"] <= 0)
                | (df["low"] <= 0)
                | (df["close"] <= 0)
            ).sum()
        ),
        "days_with_375_bars": int((bars_per_day == 375).sum()),
        "days_below_375_bars": int((bars_per_day < 375).sum()),
        "days_above_375_bars": int((bars_per_day > 375).sum()),
        "minimum_bars_in_day": int(bars_per_day.min()),
        "maximum_bars_in_day": int(bars_per_day.max()),
    })

minute_summary = pd.DataFrame(minute_summaries)

In [ ]:
print("Number of minute symbols:", minute_summary["symbol"].nunique())
print("Row count range:", minute_summary["rows"].min(), "to", minute_summary["rows"].max())
print("Trading day range:", minute_summary["trading_days"].min(), "to", minute_summary["trading_days"].max())

print("\nSymbols with duplicate timestamps:")
display(minute_summary[minute_summary["duplicate_timestamps"] > 0])

print("\nSymbols with incomplete sessions:")
display(
    minute_summary[
        (minute_summary["days_below_375_bars"] > 0)
        | (minute_summary["days_above_375_bars"] > 0)
    ][
        [
            "symbol",
            "days_below_375_bars",
            "days_above_375_bars",
            "minimum_bars_in_day",
            "maximum_bars_in_day",
        ]
    ]
)

In [ ]:
daily_symbols = set(daily_summary["symbol"])
minute_symbols = set(minute_summary["symbol"])

print("Only in daily:", sorted(daily_symbols - minute_symbols))
print("Only in minute:", sorted(minute_symbols - daily_symbols))

In [ ]:
daily_summary.to_csv("../outputs/daily_data_summary.csv", index=False)
minute_summary.to_csv("../outputs/minute_data_summary.csv", index=False)